# 0 · Configurar y probar tu entorno
- **Qué harás:** comprobar Python, paquetes y tu ruta de modelo, y hacer una petición de prueba.
- **Duración:** opcional, unos 5 min.
- **Necesitas:** el entorno instalado (`uv sync`). Si prefieres la terminal, `bash instalar.sh` hace lo mismo con preguntas guiadas.


## 1 · Python y paquetes
Debe salir Python 3.12 o superior, y ningún paquete en «falta». Si algo falta, ejecuta `uv sync` en la raíz del repositorio y, en VS Code, elige de nuevo el kernel `.venv` (*Select Kernel → Python Environments → `.venv`*) y pulsa **Restart**.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python", sys.version.split()[0], "(correcto)" if sys.version_info >= (3, 12) else "(falta 3.12+: usa `uv run`)")
for paquete in ["pydantic-ai-slim", "pydantic-settings", "httpx", "ipykernel", "rich"]:
    try:
        print(f"  {paquete} {version(paquete)}")
    except PackageNotFoundError:
        print(f"  {paquete}: FALTA (ejecuta `uv sync`)")


## 2 · Qué ruta de modelo está activa
Lee `.env` (nunca muestra las claves). Si no existe `.env`, ejecuta `uv run python -m utilidades.configurar` en una terminal. Recuerda: **`.env` gana sobre las variables de tu terminal**, y los notebooks usan el modelo simulado mientras `COURSE_OFFLINE=1`.


In [ ]:
from utilidades.config import ROOT, Settings, describir_ruta

print(".env existe:", (ROOT / ".env").exists())
s = Settings()
print("MODE =", s.mode, "· COURSE_OFFLINE =", int(s.course_offline))
print("Modelo de la ruta:", describir_ruta(s))
print("Clave de Gemini:", "sí" if s.gemini_api_key.get_secret_value() else "no")
print("Clave de OpenRouter:", "sí" if s.openrouter_api_key.get_secret_value() else "no")
if s.course_offline:
    print("Los notebooks usarán el modelo SIMULADO. Para IA real: COURSE_OFFLINE=0 en .env y reinicia el kernel.")


## 3 · Petición de prueba
Llama de verdad al modelo de tu ruta (ignora `COURSE_OFFLINE`). Con MODE=offline no hay IA que llamar.


In [ ]:
from utilidades.doctor import prueba_en_vivo

if s.mode == "offline":
    print("MODE=offline: no hay modelo real. Configura una ruta con `uv run python -m utilidades.configurar`.")
else:
    try:
        print(await prueba_en_vivo(s))
    except Exception as exc:
        print(f"{type(exc).__name__}: {str(exc)[:200]}")
        print("429 = cuota agotada (espera o cambia de modelo); 401/403/400 = clave mal copiada; ConnectError = sin internet u Ollama apagado.")


## 4 · Cómo cambiar de ruta
- **Guiado:** `uv run python -m utilidades.configurar` en la terminal (conserva lo demás de `.env`).
- **A mano:** edita `.env`: `MODE` (`gemini`, `openrouter`, `local`, `offline`), la clave y el modelo; luego reinicia el kernel.
- **Comprobar:** `uv run python -m utilidades.doctor --live`.
- Con las rutas gratuitas tus mensajes pueden usarse para mejorar los productos del proveedor: usa solo datos ficticios.
